# 37. Low-level ROOT I/O: CP trees and TH2 histograms

**Objectives:**
- Write a B+/B- pair of toy samples to one ROOT `TTree` with a signed `charge` branch using `write_cp_phase_space_sample`.
- Read that tree back with `read_root_tree` and `read_phase_space_sample` and confirm a round trip.
- Write a small ROOT `TH2` and read it back with `read_root_histogram2d`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV², daughter indices start at zero.
This is a low-level-API lesson: it does not build a fit, only exercises the ROOT I/O helpers directly.


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np
import uproot
from pathlib import Path

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    generate_toy,
    write_cp_phase_space_sample, read_root_tree, read_phase_space_sample,
    read_root_histogram2d,
)

output_dir = Path(".")


## 1. Build a small B+/B- pair of toy samples

Any two `PhaseSpaceSample` objects work for this I/O lesson; the physics only needs to be
self-consistent. We generate two small unweighted toys from the same illustrative model and
simply label them "plus" and "minus", exactly the shapes `write_cp_phase_space_sample` expects.


In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(0.5, -0.2), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=80, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

plus_sample = generate_toy(
    model, 500, parameters=truth, seed=101,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
minus_sample = generate_toy(
    model, 300, parameters=truth, seed=202,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
print(f"B+ toy: {plus_sample.size} events, B- toy: {minus_sample.size} events")


B+ toy: 500 events, B- toy: 300 events


## 2. Write both charges to one TTree

`write_cp_phase_space_sample` writes `charge=+1` for the first sample and `charge=-1` for the
second into one common tree (default name `DecayTree`), the same signed-`charge` convention used
by `generate_cp_toy`'s `output_root=` option (see `docs/toy_generation.md`).


In [3]:
cp_tree_path = output_dir / "tutorial_37_cp_tree.root"
write_cp_phase_space_sample(
    cp_tree_path, plus_sample, minus_sample,
    tree="DecayTree", charge_branch="charge", include_momenta=False,
)
print("Wrote", cp_tree_path, "size:", cp_tree_path.stat().st_size, "bytes")


Wrote tutorial_37_cp_tree.root size: 42770 bytes


## 3. Read the tree back and confirm the round trip

`read_root_tree` reads arbitrary branches (here including `charge`); `read_phase_space_sample`
builds a `PhaseSpaceSample` from the invariant branches. Selecting on `charge` recovers each
original sample exactly, and `uproot`'s own `cut=` mirrors the selection shown in
`docs/toy_generation.md`.


In [4]:
raw = read_root_tree(
    cp_tree_path, "DecayTree",
    {"s12": "s12", "s13": "s13", "s23": "s23", "charge": "charge"},
)
assert raw["charge"].shape[0] == plus_sample.size + minus_sample.size

loaded_plus = read_phase_space_sample(
    cp_tree_path, "DecayTree", s12="s12", s13="s13", s23="s23", cut="charge > 0",
)
loaded_minus = read_phase_space_sample(
    cp_tree_path, "DecayTree", s12="s12", s13="s13", s23="s23", cut="charge < 0",
)
np.testing.assert_allclose(np.asarray(loaded_plus.s12), np.asarray(plus_sample.s12))
np.testing.assert_allclose(np.asarray(loaded_minus.s12), np.asarray(minus_sample.s12))
assert loaded_plus.size == plus_sample.size
assert loaded_minus.size == minus_sample.size
print("Round trip confirmed: B+ and B- invariants match the generated samples exactly.")


Round trip confirmed: B+ and B- invariants match the generated samples exactly.


## 4. Writing and reading a ROOT TH2

`dalitzplotfitter` only *reads* ROOT histograms (`read_root_histogram2d` and the
`histogram_efficiency_from_root`/`square_dalitz_*_from_root` helpers built on it, see
`docs/root_io.md`) — it has no TH2 writer of its own, since production efficiency/background maps
are normally produced by an external ROOT-based tool, not by this package. To exercise
`read_root_histogram2d` here without depending on another notebook, we write the TH2 with `uproot`
directly: `uproot` accepts a `(values, x_edges, y_edges)` tuple assigned to a file key and encodes
it as a native `TH2D`, so no PyROOT is needed on either side of the round trip.


In [5]:
rng = np.random.default_rng(7)
x_edges = np.linspace(0.2, 1.6, 7)
y_edges = np.linspace(0.2, 1.6, 5)
values = rng.uniform(0.8, 1.2, size=(len(x_edges) - 1, len(y_edges) - 1))

hist_path = output_dir / "tutorial_37_histogram.root"
with uproot.recreate(hist_path) as f:
    f["efficiency_map"] = (values, x_edges, y_edges)

read_values, read_x_edges, read_y_edges = read_root_histogram2d(hist_path, "efficiency_map")
np.testing.assert_allclose(np.asarray(read_values), values)
np.testing.assert_allclose(np.asarray(read_x_edges), x_edges)
np.testing.assert_allclose(np.asarray(read_y_edges), y_edges)
print("TH2 round trip confirmed:", read_values.shape, "bins")


TH2 round trip confirmed: (6, 4) bins


## Summary and exercises

1. `write_cp_phase_space_sample` and the signed `charge` branch are exactly what `generate_cp_toy`
   uses internally for its own `output_root=` option.
2. `read_root_histogram2d` returns `(values, x_edges, y_edges)` with over/underflow excluded; it is
   the building block behind `histogram_efficiency_from_root` and the Square-Dalitz histogram
   readers.
3. Try `include_momenta=True` in step 1 and confirm the extra `p1_E`/`p1_PX`/... branches round-trip too.
4. Build a `HistogramEfficiency` directly from the file written in section 4 with
   `histogram_efficiency_from_root(hist_path, "efficiency_map", x_variable="s12", y_variable="s13")`.

Reference: [ROOT I/O](../../docs/root_io.md), [toy generation and ROOT output](../../docs/toy_generation.md).

Return to [the course guide](TUTORIALS.md).
